### Bielik w systemach agentowych - dodatek "Bielik w Google Colab"

<img src="https://bielik.ai/wp-content/uploads/2024/08/Bielik_Secondary_Rabarbar-300x82.webp" width=350>

Autor: **Piotr Tynecki** | Aktualizacja: **21.11.2025**

In [ ]:
!pip install -q transformers torch accelerate sentencepiece

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Używanie urządzenia: {device}")

In [ ]:
DEVICE = torch.device("cuda")

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

print("Zalogowano do Hugging Face")

In [ ]:
MODEL_NAME = "speakleash/Bielik-4.5B-v3.0-Instruct"

In [ ]:
# Ładowanie tokenizera - zasady konwersji tekstu na tokeny
print("Ładowanie tokenizera...\n")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
# Ładowanie modelu z precyzją bfloat16
print("Ładowanie modelu (to może potrwać kilka minut)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    device_map="auto"
)

In [ ]:
print(f"[INFO] Całkowita pamięć vRAM zajęta przez model: {model.get_memory_footprint() / (1024 ** 3):.2f} GB")

In [ ]:
user_prompt = "Przepis na kartoszki w trzech zdaniach"

In [ ]:
messages = [
    {"role": "system", "content": "Odpowiadaj krótko, precyzyjnie i wyłącznie w języku polskim."},
    {"role": "user", "content": user_prompt}
]

In [ ]:
input_ids = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt"
).to(DEVICE)

In [ ]:
outputs = model.generate(
    input_ids,
    # Maksymalna liczba nowych tokenów, które model może wygenerować,
    # zapobiegając niekontrolowanemu generowaniu tekstu.
    max_new_tokens=1000,
)

In [ ]:
response = tokenizer.batch_decode(outputs)
print(response[0])